In [6]:
# Cell 1: portable project setup
import importlib
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / ".env").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
        nested_classifier = candidate / "classifier"
        if (
            (nested_classifier / ".env").is_file()
            and (nested_classifier / "pyproject.toml").is_file()
        ):
            return nested_classifier
    raise FileNotFoundError(
        "Не найден корень classificator: нужна папка с .env и pyproject.toml"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SOURCE_ROOT = PROJECT_ROOT / "src"
os.chdir(PROJECT_ROOT)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

BATCH_SIZE = 30
BATCH_INDEX = 2  # starts at cases [30:60]; use 0 for [0:30]
MAX_CASES = 300
CASE_OFFSET = BATCH_INDEX * BATCH_SIZE
MAX_WORKERS = 8
REBUILD_MANIFEST = False
AUTO_START_BATCH_INDEX = BATCH_INDEX
AUTO_BATCH_COUNT = None  # None = process all batches up to MAX_CASES
BATCH_TAG = f"batch_{BATCH_INDEX:04d}"
print("project root:", PROJECT_ROOT)
print(
    "batch:",
    BATCH_TAG,
    "offset:",
    CASE_OFFSET,
    "size:",
    BATCH_SIZE,
    "workers:",
    MAX_WORKERS,
    "auto start:",
    AUTO_START_BATCH_INDEX,
    "auto count:",
    AUTO_BATCH_COUNT,
)


project root: /Users/zero/GITHUB/vue-graph-rag/classifier
batch: batch_0002 offset: 60 size: 30 workers: 8 auto start: 2 auto count: None


In [7]:
# Cell 2: importlib для каждого production-модуля
import importlib


def load_module(name: str, reload: bool = True):
    module = importlib.import_module(name)
    if reload:
        module = importlib.reload(module)
    print(f"{name:45} -> {module.__file__}")
    return module


config = load_module("telegram_classificator.conf", reload=True)
config.configure_logging()
qa_parser = load_module("telegram_classificator.qa_parser", reload=True)
image_describer = load_module("telegram_classificator.image_describer", reload=True)
promt = load_module("telegram_classificator.promt", reload=True)
agent = load_module("telegram_classificator.agent", reload=True)
print(config.summary())


telegram_classificator.conf                   -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/conf.py
telegram_classificator.qa_parser              -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/qa_parser.py
telegram_classificator.image_describer        -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/image_describer.py
telegram_classificator.promt                  -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/promt.py
telegram_classificator.agent                  -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/agent.py
{'PROJECT_ROOT': '/Users/zero/GITHUB/vue-graph-rag/classifier', 'QA_JSON': '/Users/zero/GITHUB/vue-graph-rag/classifier/qa.json', 'IMAGE_DIR': '/Users/zero/GITHUB/vue-graph-rag/classifier', 'DATA_DIR': '/Users/zero/GITHUB/vue-graph-rag/classifier/data', 'TEXT_MODEL': 'gemma4:latest', 'VISION_MODEL': 'qwen3-vl:8b', 'VISION_BASE_URL': 'http:/

In [8]:
# Cell 3: deterministic manifest and one batch of local qa.json
import json


def discover_qa_files(input_path: Path) -> list[Path]:
    if input_path.is_file():
        return [input_path]
    if input_path.is_dir():
        return sorted(input_path.rglob("qa.json"))
    raise FileNotFoundError(f"QA path not found: {input_path}")


qa_input = config.QA_JSON
if not qa_input.exists():
    qa_input = config.DATA_DIR
    print(
        "configured QA_JSON was not found; discovering qa.json under:",
        qa_input,
    )

qa_files = discover_qa_files(qa_input)
if not qa_files:
    raise FileNotFoundError(f"No qa.json files found under: {qa_input}")

MASTER_CASES = config.DATA_DIR / "cases_manifest.jsonl"
if MASTER_CASES.exists() and not REBUILD_MANIFEST:
    all_cases = [
        json.loads(line)
        for line in MASTER_CASES.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print("loaded master manifest:", MASTER_CASES)
    print("manifest total:", len(all_cases))
else:
    all_cases = []
    for qa_file in qa_files:
        print(f"parsing {qa_file.parent.name}/qa.json")
        parsed_cases = qa_parser.qa_to_cases(
            qa_file,
            case_id_prefix=qa_file.parent.name,
        )
        all_cases.extend(parsed_cases)
        print("parsed cases:", len(parsed_cases), "manifest total:", len(all_cases))
    qa_parser.write_jsonl(all_cases, MASTER_CASES)
    print("master manifest written:", MASTER_CASES)

batch_start = CASE_OFFSET
processing_limit = min(MAX_CASES, len(all_cases))
batch_end = min(CASE_OFFSET + BATCH_SIZE, processing_limit)
cases = all_cases[batch_start:batch_end]
if not cases and all_cases:
    raise IndexError(
        f"batch offset {CASE_OFFSET} is outside processing range 0:{processing_limit}"
    )

BATCH_CASES = config.DATA_DIR / f"cases_{BATCH_TAG}.jsonl"
BATCH_IMAGES = config.DATA_DIR / f"image_descriptions_{BATCH_TAG}.jsonl"
BATCH_IMAGE_ERRORS = config.DATA_DIR / f"image_errors_{BATCH_TAG}.jsonl"
BATCH_RESULTS = config.DATA_DIR / f"results_{BATCH_TAG}.jsonl"
BATCH_CLASS_ERRORS = config.DATA_DIR / f"classification_errors_{BATCH_TAG}.jsonl"
BATCH_ERRORS = config.DATA_DIR / f"errors_{BATCH_TAG}.jsonl"
BATCH_MANIFEST = config.DATA_DIR / f"manifest_{BATCH_TAG}.json"

qa_parser.write_jsonl(cases, BATCH_CASES)
manifest = {
    "status": "parsed",
    "batch_index": BATCH_INDEX,
    "batch_tag": BATCH_TAG,
    "batch_size": BATCH_SIZE,
    "max_cases": MAX_CASES,
    "case_offset": CASE_OFFSET,
    "total_cases": len(all_cases),
    "processable_cases": processing_limit,
    "selected_cases": len(cases),
    "remaining_cases": max(0, processing_limit - batch_end),
    "qa_files_discovered": len(qa_files),
    "artifact_paths": {
        "master_cases": str(MASTER_CASES),
        "cases": str(BATCH_CASES),
        "images": str(BATCH_IMAGES),
        "image_errors": str(BATCH_IMAGE_ERRORS),
        "results": str(BATCH_RESULTS),
        "classification_errors": str(BATCH_CLASS_ERRORS),
        "errors": str(BATCH_ERRORS),
    },
}
BATCH_MANIFEST.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("qa input:", qa_input)
print("qa files discovered:", len(qa_files))
print("manifest total:", len(all_cases))
print("processing limit:", processing_limit)
print("batch selected:", len(cases), f"[{CASE_OFFSET}:{batch_end}]")
print("cases output:", BATCH_CASES)
print("manifest output:", BATCH_MANIFEST)


configured QA_JSON was not found; discovering qa.json under: /Users/zero/GITHUB/vue-graph-rag/classifier/data
loaded master manifest: /Users/zero/GITHUB/vue-graph-rag/classifier/data/cases_manifest.jsonl
manifest total: 1944
qa input: /Users/zero/GITHUB/vue-graph-rag/classifier/data
qa files discovered: 132
manifest total: 1944
processing limit: 300
batch selected: 30 [60:90]
cases output: /Users/zero/GITHUB/vue-graph-rag/classifier/data/cases_batch_0002.jsonl
manifest output: /Users/zero/GITHUB/vue-graph-rag/classifier/data/manifest_batch_0002.json


In [9]:
# Cell 4: image descriptions for the selected batch
image_occurrences = [image for case in cases for image in case.get("images", [])]
image_unique_paths = {
    image.get("abs_path")
    for image in image_occurrences
    if image.get("exists")
}
print(
    "image stage starting:",
    len(image_occurrences),
    "occurrences,",
    len(image_unique_paths),
    "unique files, workers=",
    MAX_WORKERS,
)
try:
    image_descriptions = image_describer.describe_case_images(
        cases,
        model=config.VISION_MODEL,
        base_url=config.VISION_BASE_URL,
        sentences=5,
        max_workers=MAX_WORKERS,
    )
except Exception as error:
    manifest["status"] = "failed"
    manifest["failed_stage"] = "image_description"
    manifest["failure"] = str(error)[:1000]
    BATCH_MANIFEST.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    raise
image_describer.write_descriptions(image_descriptions, BATCH_IMAGES)
image_describer.write_errors(image_descriptions, BATCH_IMAGE_ERRORS)

shown_images = 0
for case_id, images in image_descriptions.items():
    for image in images:
        print("case:", case_id)
        print("case:", case_id)
        print("path:", image.get("relative_path"))
        print("visual_summaemary:", image.get("visual_summary"))
        print("detailed_description:", image.get("detailed_description"))
        print("desc_error:", image.get("desc_error"))
        print("---")
        shown_images += 1
        if shown_images >= 10:
            break
    if shown_images >= 10:
        break
print("image descriptions output:", BATCH_IMAGES)
print("image errors output:", BATCH_IMAGE_ERRORS)


2026-09-23 17:28:31,344 INFO telegram_classificator.image_describer image stage started occurrences=26 unique=26 pending=26 workers=8
2026-09-23 17:28:31,345 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/434110464.jpg
2026-09-23 17:28:31,346 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/284164096.jpg
2026-09-23 17:28:31,346 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/285212672.jpg
2026-09-23 17:28:31,347 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/286261248.jpg
2026-09-23 17:28:31,347 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/113246208.jpg
2026-09-23 17:28:31,348 INFO telegram_classificator.image_describer image description started path=images/veu_support/-100390483985

image stage starting: 26 occurrences, 26 unique files, workers= 8


describe images:   0%|          | 0/26 [00:00<?, ?image/s]2026-09-23 17:28:55,901 INFO telegram_classificator.image_describer image description finished path=images/veu_support/-1003904839852/434110464.jpg status=ok seconds=24.6
2026-09-23 17:28:55,906 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/374341632.jpg
describe images:   4%|▍         | 1/26 [00:24<10:13, 24.52s/image]2026-09-23 17:28:55,907 INFO telegram_classificator.image_describer image stage progress completed=1/26 path=images/veu_support/-1003904839852/434110464.jpg status=ok
2026-09-23 17:29:16,931 INFO telegram_classificator.image_describer image description finished path=images/veu_support/-1003904839852/284164096.jpg status=ok seconds=45.6
2026-09-23 17:29:16,932 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/394264576.jpg
describe images:   8%|▊         | 2/26 [00:45<08:59, 22.47s/image]2026-09

case: aleksandr_komlev__lex_freeman__q_0011
case: aleksandr_komlev__lex_freeman__q_0011
path: images/veu_support/-1003904839852/434110464.jpg
visual_summaemary: Фрагмент инструкции о самостоятельной подготовке архитектурных рассказов и технической части легенды.
detailed_description: Это скриншот части учебной инструкции по изучению технических аспектов продуктов. Контекст — обучение работе с архитектурой программных продуктов и подготовка к техническим интервью. Важно, что инструкция подчеркивает необходимость визуализации архитектуры через инструменты вроде МИРО и обращения к тех-ментору при сложностях. Документ описывает этапы изучения параллельных софт-частей, подготовки к интервью и работы с документацией. Инструкция фокусируется на подготовке к ответам на вопросы HR и технических интервью, включая анализ деталей архитектуры.
desc_error: None
---
case: aleksandr_komlev__lex_freeman__q_0011
case: aleksandr_komlev__lex_freeman__q_0011
path: images/veu_support/-1003904839852/28416409

In [10]:
# Cell 5: classification, error inventory and batch manifest

def read_jsonl_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


try:
    agent.run(
        BATCH_CASES,
        BATCH_RESULTS,
        img_desc_path=BATCH_IMAGES,
        limit=None,
        max_workers=MAX_WORKERS,
        errors_path=BATCH_CLASS_ERRORS,
    )
except Exception as error:
    manifest["status"] = "failed"
    manifest["failure"] = str(error)[:1000]
    BATCH_MANIFEST.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    raise

image_error_rows = read_jsonl_rows(BATCH_IMAGE_ERRORS)
classification_error_rows = read_jsonl_rows(BATCH_CLASS_ERRORS)
all_error_rows = image_error_rows + classification_error_rows
with BATCH_ERRORS.open("w", encoding="utf-8") as error_file:
    for row in all_error_rows:
        error_file.write(json.dumps(row, ensure_ascii=False) + "\n")

result_rows = read_jsonl_rows(BATCH_RESULTS)
manifest.update(
    {
        "status": "completed",
        "successful_cases": sum(1 for row in result_rows if not row.get("error")),
        "classification_errors": len(classification_error_rows),
        "image_errors": len(image_error_rows),
        "total_errors": len(all_error_rows),
    }
)
BATCH_MANIFEST.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("classification output:", BATCH_RESULTS)
print("classification errors:", BATCH_CLASS_ERRORS)
print("all errors:", BATCH_ERRORS)
print("batch manifest:", BATCH_MANIFEST)
print(
    "batch summary:",
    "successful=", manifest["successful_cases"],
    "errors=", manifest["total_errors"],
    "remaining=", manifest["remaining_cases"],
)


2026-09-23 17:38:17,131 INFO telegram_classificator.agent start cases=30 model=gemma4:latest workers=8
2026-09-23 17:38:17,132 INFO telegram_classificator.agent classification stage started cases=30 workers=8
2026-09-23 17:38:17,132 INFO telegram_classificator.agent case started case=aleksandr_komlev__lex_freeman__q_0007 model=gemma4:latest
2026-09-23 17:38:17,132 INFO telegram_classificator.agent case started case=aleksandr_komlev__lex_freeman__q_0008 model=gemma4:latest
classify:   0%|          | 0/30 [00:00<?, ?case/s]2026-09-23 17:38:17,133 INFO telegram_classificator.agent case started case=aleksandr_komlev__lex_freeman__q_0009 model=gemma4:latest
2026-09-23 17:38:17,133 INFO telegram_classificator.agent case started case=aleksandr_komlev__lex_freeman__q_0010 model=gemma4:latest
2026-09-23 17:38:17,133 INFO telegram_classificator.agent case started case=aleksandr_komlev__lex_freeman__q_0011 model=gemma4:latest
2026-09-23 17:38:17,133 INFO telegram_classificator.agent case started 

ModelAPIError: text API request failed; inspect OPENAI_BASE_URL and MODEL

In [ ]:
# Cell 6: automatically process missing batches
from tqdm.auto import tqdm


def _read_jsonl_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def _batch_artifacts(batch_index: int) -> dict[str, Path]:
    tag = f"batch_{batch_index:04d}"
    return {
        "cases": config.DATA_DIR / f"cases_{tag}.jsonl",
        "images": config.DATA_DIR / f"image_descriptions_{tag}.jsonl",
        "image_errors": config.DATA_DIR / f"image_errors_{tag}.jsonl",
        "results": config.DATA_DIR / f"results_{tag}.jsonl",
        "classification_errors": config.DATA_DIR / f"classification_errors_{tag}.jsonl",
        "errors": config.DATA_DIR / f"errors_{tag}.jsonl",
        "manifest": config.DATA_DIR / f"manifest_{tag}.json",
    }


def _write_batch_manifest(manifest: dict, path: Path) -> None:
    path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def _process_batch(batch_index: int) -> dict:
    artifacts = _batch_artifacts(batch_index)
    batch_start = batch_index * BATCH_SIZE
    processing_limit = min(MAX_CASES, len(all_cases))
    batch_end = min(batch_start + BATCH_SIZE, processing_limit)
    cases_for_batch = all_cases[batch_start:batch_end]
    if not cases_for_batch:
        raise IndexError(
            f"batch index {batch_index} is outside processing range 0:{processing_limit}"
        )

    manifest_path = artifacts["manifest"]
    if manifest_path.exists():
        existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        required_outputs = tuple(
            artifacts[name]
            for name in (
                "cases",
                "images",
                "image_errors",
                "results",
                "classification_errors",
                "errors",
            )
        )
        if existing_manifest.get("status") == "completed" and all(
            path.exists() for path in required_outputs
        ):
            print(
                f"skip completed batch={batch_index} "
                f"range=[{batch_start}:{batch_end}]"
            )
            return existing_manifest

    manifest = {
        "status": "parsed",
        "batch_index": batch_index,
        "batch_tag": f"batch_{batch_index:04d}",
        "batch_size": BATCH_SIZE,
        "max_cases": MAX_CASES,
        "case_offset": batch_start,
        "total_cases": len(all_cases),
        "processable_cases": processing_limit,
        "selected_cases": len(cases_for_batch),
        "remaining_cases": max(0, processing_limit - batch_end),
        "artifact_paths": {
            "master_cases": str(MASTER_CASES),
            **{name: str(path) for name, path in artifacts.items()},
        },
    }
    qa_parser.write_jsonl(cases_for_batch, artifacts["cases"])
    _write_batch_manifest(manifest, manifest_path)
    print(
        f"start batch={batch_index} "
        f"range=[{batch_start}:{batch_end}] cases={len(cases_for_batch)}"
    )

    try:
        descriptions = image_describer.describe_case_images(
            cases_for_batch,
            model=config.VISION_MODEL,
            base_url=config.VISION_BASE_URL,
            sentences=5,
            max_workers=MAX_WORKERS,
        )
        image_describer.write_descriptions(descriptions, artifacts["images"])
        image_describer.write_errors(descriptions, artifacts["image_errors"])
    except Exception as error:
        manifest.update(
            {
                "status": "failed",
                "failed_stage": "image_description",
                "failure": str(error)[:1000],
            }
        )
        _write_batch_manifest(manifest, manifest_path)
        raise

    try:
        agent.run(
            artifacts["cases"],
            artifacts["results"],
            img_desc_path=artifacts["images"],
            limit=None,
            max_workers=MAX_WORKERS,
            errors_path=artifacts["classification_errors"],
        )
    except Exception as error:
        manifest.update(
            {
                "status": "failed",
                "failed_stage": "classification",
                "failure": str(error)[:1000],
            }
        )
        _write_batch_manifest(manifest, manifest_path)
        raise

    image_errors = _read_jsonl_rows(artifacts["image_errors"])
    classification_errors = _read_jsonl_rows(artifacts["classification_errors"])
    all_errors = image_errors + classification_errors
    with artifacts["errors"].open("w", encoding="utf-8") as error_file:
        for row in all_errors:
            error_file.write(json.dumps(row, ensure_ascii=False) + "\n")

    result_rows = _read_jsonl_rows(artifacts["results"])
    manifest.update(
        {
            "status": "completed",
            "successful_cases": sum(
                1 for row in result_rows if not row.get("error")
            ),
            "classification_errors": len(classification_errors),
            "image_errors": len(image_errors),
            "total_errors": len(all_errors),
        }
    )
    _write_batch_manifest(manifest, manifest_path)
    print(
        f"completed batch={batch_index} "
        f"success={manifest['successful_cases']} "
        f"errors={manifest['total_errors']}"
    )
    return manifest


if AUTO_START_BATCH_INDEX < 0:
    raise ValueError("AUTO_START_BATCH_INDEX must be non-negative")
if AUTO_BATCH_COUNT is not None and AUTO_BATCH_COUNT < 0:
    raise ValueError("AUTO_BATCH_COUNT must be non-negative or None")

last_batch_index = (min(MAX_CASES, len(all_cases)) - 1) // BATCH_SIZE
if AUTO_BATCH_COUNT is None:
    auto_end_batch = last_batch_index + 1
else:
    auto_end_batch = min(
        AUTO_START_BATCH_INDEX + AUTO_BATCH_COUNT,
        last_batch_index + 1,
    )
if AUTO_START_BATCH_INDEX > auto_end_batch:
    raise ValueError("AUTO_START_BATCH_INDEX is outside the selected batch range")

batch_indexes = range(AUTO_START_BATCH_INDEX, auto_end_batch)
print(
    "auto batch range:",
    f"[{AUTO_START_BATCH_INDEX}:{auto_end_batch})",
    "case range:",
    f"[{AUTO_START_BATCH_INDEX * BATCH_SIZE}:{min(auto_end_batch * BATCH_SIZE, MAX_CASES)}]",
)
for batch_index in tqdm(batch_indexes, desc="auto batches", unit="batch"):
    _process_batch(batch_index)
print("automatic batch processing finished")


auto batch range: [2:10) case range: [60:300]


auto batches:   0%|          | 0/8 [00:00<?, ?batch/s]2026-09-23 18:22:47,567 INFO telegram_classificator.image_describer image stage started occurrences=26 unique=26 pending=26 workers=8
2026-09-23 18:22:47,568 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/434110464.jpg
2026-09-23 18:22:47,568 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/284164096.jpg
2026-09-23 18:22:47,569 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/285212672.jpg


start batch=2 range=[60:90] cases=30


2026-09-23 18:22:47,569 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/286261248.jpg

describe images:   0%|          | 0/26 [00:00<?, ?image/s]2026-09-23 18:22:47,570 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/113246208.jpg
2026-09-23 18:22:47,571 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/157286400.jpg
2026-09-23 18:22:47,571 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/163577856.jpg
2026-09-23 18:22:47,571 INFO telegram_classificator.image_describer image description started path=images/veu_support/-1003904839852/395313152.jpg
2026-09-23 18:23:09,315 INFO telegram_classificator.image_describer image description finished path=images/veu_support/-1003904839852/395313152.jpg status=ok seconds=21.7
2026-09-23 18:23:09,316 INFO telegr